# 12. Upload do dataset para o Hugging Face Hub

Este notebook prepara e envia para o Hugging Face Hub os datasets e views gerados pelo experimento de avaliação de defesas contra prompt injection.

A ideia deste notebook é parecida com o notebook 11, mas aplicada ao **dataset experimental**, não aos adaptadores treinados. O repositório criado no Hugging Face deve ser do tipo `Dataset`.

O conteúdo enviado inclui os arquivos JSONL canônicos, as views de treinamento e avaliação, manifestos leves e um dataset card (`README.md`) explicando o propósito, a estrutura e as limitações do dataset.

Este notebook foi desenhado com uma restrição importante de espaço em disco:

```text
Não copiar datasets para exports/.
Não criar staging pesado.
Enviar os arquivos diretamente dos diretórios originais.
Usar exports/ apenas para metadados leves gerados localmente.
```

Isso evita duplicar arquivos grandes dentro do projeto. O diretório `exports/huggingface_dataset_upload/` conterá apenas arquivos pequenos, como README, manifesto e índices de upload.


## 1. Escopo do upload

Este notebook envia para o Hugging Face Hub os artefatos de dataset necessários para reproduzir a etapa de dados do experimento.

Por padrão, o upload inclui:

```text
configs/
manifests/data/
manifests/environment/
data/canonical/
data/views/
```

O diretório `data/canonical/` contém os arquivos em formato canônico, incluindo exemplos limpos e exemplos atacados. O diretório `data/views/` contém as versões específicas usadas por cada cenário experimental, como StruQ-like SFT, SecAlign-like DPO, Instruction-Hierarchy-like SFT e arquivos comuns de avaliação.

Este notebook **não** envia:

```text
data/cache/
.venv/
adapters/
outputs de modelo
logs de treinamento
resultados de inferência
resultados de métricas
```

Esses itens pertencem a outras partes do projeto. Os adaptadores são tratados pelo notebook 11. Os resultados e logs completos são tratados pelo notebook 10 de exportação dos artefatos.


## 2. Observação sobre conteúdo sensível

Este dataset pode conter conteúdo sensível ou ofensivo porque uma das tasks usadas no experimento é `hsol`, associada à classificação de linguagem ofensiva, discurso de ódio ou conteúdo neutro.

Além disso, os arquivos atacados contêm templates de prompt injection, incluindo instruções maliciosas simuladas dentro de dados não confiáveis. Esses ataques foram criados para fins de avaliação de robustez, não para uso operacional.

Por isso, recomenda-se que o repositório seja criado inicialmente como **privado**. Depois do upload, o dataset card deve ser revisado manualmente antes de qualquer decisão de tornar o repositório público.

Antes de executar um upload real, este notebook exige uma confirmação explícita:

```python
ACKNOWLEDGE_DATASET_CONTENT = True
```

Essa confirmação serve para lembrar que o dataset pode conter texto ofensivo real e exemplos de prompt injection gerados para pesquisa.


## 3. Imports e configuração de caminhos

Nesta etapa, são definidos os imports principais e os caminhos usados pelo notebook.

A raiz esperada do projeto é:

```text
/workspace/pi-defense-exp
```

Os metadados leves gerados por este notebook serão salvos em:

```text
exports/huggingface_dataset_upload/
logs/huggingface_dataset_upload/
manifests/huggingface_dataset_upload/
```

Essas pastas não recebem cópias dos arquivos de dataset. Elas guardam apenas documentação, índices e manifestos do processo de upload.


In [1]:
import json
import os
import sys
import time
import traceback
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd


In [2]:
PROJECT_ROOT = Path("/workspace/pi-defense-exp")
EXPECTED_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"

CONFIG_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
CANONICAL_DIR = DATA_DIR / "canonical"
VIEWS_DIR = DATA_DIR / "views"
MANIFESTS_DIR = PROJECT_ROOT / "manifests"

HF_DATASET_EXPORT_DIR = PROJECT_ROOT / "exports" / "huggingface_dataset_upload"
HF_DATASET_LOG_DIR = PROJECT_ROOT / "logs" / "huggingface_dataset_upload"
HF_DATASET_MANIFEST_DIR = PROJECT_ROOT / "manifests" / "huggingface_dataset_upload"

for path in [HF_DATASET_EXPORT_DIR, HF_DATASET_LOG_DIR, HF_DATASET_MANIFEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python atual:", sys.executable)
print("Python esperado:", EXPECTED_PYTHON)
print("Diretório de metadados leves:", HF_DATASET_EXPORT_DIR)


Project root: /workspace/pi-defense-exp
Python atual: /workspace/pi-defense-exp/.venv/bin/python
Python esperado: /workspace/pi-defense-exp/.venv/bin/python
Diretório de metadados leves: /workspace/pi-defense-exp/exports/huggingface_dataset_upload


## 4. Funções utilitárias

As funções desta seção são usadas para ler e escrever JSON, registrar eventos, contar linhas JSONL e criar índices dos arquivos que serão enviados.

O log deste notebook é incremental. Isso significa que eventos importantes são registrados conforme acontecem, em vez de apenas no final. Essa estratégia ajuda a diagnosticar falhas de autenticação, falta de arquivos ou interrupções durante o upload.


In [3]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sanitize_json_value(value):
    if isinstance(value, dict):
        return {str(k): sanitize_json_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_json_value(v) for v in value]
    if isinstance(value, tuple):
        return [sanitize_json_value(v) for v in value]
    try:
        if pd.isna(value):
            return "NaN"
    except Exception:
        pass
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sanitize_json_value(data), f, indent=2, ensure_ascii=False, allow_nan=False)


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(sanitize_json_value(row), ensure_ascii=False, allow_nan=False) + "\n")


def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)


def log_event(event_type: str, payload: dict | None = None) -> None:
    event = {"timestamp_utc": utc_now(), "event_type": event_type}
    if payload:
        event.update(payload)
    append_jsonl(HF_DATASET_LOG_DIR / "12_upload_dataset_to_huggingface_events.jsonl", event)


## 5. Configuração do repositório de dataset

Nesta etapa, são definidos o namespace, o nome do repositório e o modo de execução.

Por segurança, o notebook começa em modo:

```python
DRY_RUN = True
```

Nesse modo, o notebook valida os arquivos e mostra o que seria enviado, mas não cria repositório e não faz upload.

Para executar o upload real, altere:

```python
DRY_RUN = False
```

Também é recomendado começar com:

```python
PRIVATE_REPO = True
```

Isso permite revisar o dataset card, os arquivos enviados e a estrutura do repositório antes de decidir se o dataset deve ser público.


In [4]:
# Altere estes campos antes do upload real.
HF_NAMESPACE = "leinha"
DATASET_REPO_NAME = "pi-defense-experiment-dataset"
DATASET_REPO_ID = f"{HF_NAMESPACE}/{DATASET_REPO_NAME}"

DRY_RUN = False
PRIVATE_REPO = True

# Confirmação explícita para upload real.
# Mude para True apenas se você entende que o dataset pode conter HSOL e templates de prompt injection.
ACKNOWLEDGE_DATASET_CONTENT = True

# Escopo do upload.
UPLOAD_CANONICAL_DATA = True
UPLOAD_DATA_VIEWS = True
UPLOAD_CONFIGS = True
UPLOAD_DATA_MANIFESTS = True

# Itens que devem permanecer fora deste upload.
UPLOAD_DATA_CACHE = False
UPLOAD_MODEL_ADAPTERS = False
UPLOAD_RESULTS = False
UPLOAD_LOGS = False

print("Dataset repo id:", DATASET_REPO_ID)
print("DRY_RUN:", DRY_RUN)
print("PRIVATE_REPO:", PRIVATE_REPO)


Dataset repo id: leinha/pi-defense-experiment-dataset
DRY_RUN: False
PRIVATE_REPO: True


## 6. Checagens de segurança antes do upload

Esta seção impede alguns erros comuns:

```text
- tentar subir cache de dados;
- tentar subir adaptadores no repositório de dataset;
- fazer upload real sem reconhecer conteúdo sensível;
- usar o namespace placeholder sem trocar para usuário/organização real.
```

Essas validações são especialmente importantes porque este notebook deve operar sem cópia intermediária. Ou seja, se uma pasta errada for selecionada, o upload partirá diretamente da fonte original.


In [5]:
if HF_NAMESPACE == "SEU_USUARIO_OU_ORG" and not DRY_RUN:
    raise ValueError("Antes do upload real, defina HF_NAMESPACE com seu usuário ou organização no Hugging Face.")

if not DRY_RUN and not ACKNOWLEDGE_DATASET_CONTENT:
    raise ValueError(
        "Para upload real, defina ACKNOWLEDGE_DATASET_CONTENT=True. "
        "Isso confirma que você entende que o dataset pode conter texto sensível/ofensivo e templates de prompt injection."
    )

if UPLOAD_DATA_CACHE:
    raise ValueError("UPLOAD_DATA_CACHE deve permanecer False para evitar upload de cache pesado.")
if UPLOAD_MODEL_ADAPTERS:
    raise ValueError("Adaptadores devem ser enviados pelo notebook 11, não pelo notebook de dataset.")
if UPLOAD_RESULTS or UPLOAD_LOGS:
    raise ValueError("Resultados e logs pertencem ao notebook 10 de exportação de artefatos, não ao repo de dataset.")

print("Checagens de segurança concluídas.")


Checagens de segurança concluídas.


## 7. Autenticação no Hugging Face

O upload para o Hub requer autenticação. Esta célula usa a biblioteca `huggingface_hub` para verificar se já existe login válido no ambiente. Se não houver, solicita o token usando `getpass`, para evitar imprimir o token no notebook.

O token precisa ter permissão de escrita para criar repositório e enviar arquivos.


In [6]:
from huggingface_hub import HfApi, create_repo, login, upload_file, upload_folder, whoami

try:
    hf_user = whoami()
    print("Login Hugging Face detectado.")
    print("User:", hf_user.get("name"))
except Exception:
    if DRY_RUN:
        print("Login não detectado, mas DRY_RUN=True. O upload real exigirá autenticação.")
    else:
        hf_token = getpass("Cole seu token do Hugging Face: ")
        login(token=hf_token, add_to_git_credential=False)
        hf_user = whoami()
        print("Login realizado.")
        print("User:", hf_user.get("name"))


Login Hugging Face detectado.
User: leinha


## 8. Definição das fontes que serão enviadas

Nesta etapa, o notebook define quais pastas e arquivos serão considerados para upload.

A regra principal é:

```text
source_path = caminho real no projeto
target_path = caminho dentro do repositório Hugging Face
```

Nada é copiado para `exports/` antes do upload. A função `upload_folder` lê diretamente de `source_path` e envia para `target_path` no repositório remoto.


In [7]:
source_folders = []
missing_sources = []


def add_source_folder(source_path: Path, target_path: str, required: bool = True, description: str = ""):
    source_path = source_path.resolve()
    if source_path.exists() and source_path.is_dir():
        source_folders.append({
            "source_path": source_path,
            "target_path": target_path.strip("/"),
            "required": required,
            "description": description,
        })
    else:
        missing_sources.append({
            "source_path": str(source_path),
            "target_path": target_path.strip("/"),
            "required": required,
            "description": description,
            "reason": "source_missing",
        })
        if required:
            raise FileNotFoundError(f"Fonte obrigatória ausente: {source_path}")


if UPLOAD_CANONICAL_DATA:
    add_source_folder(CANONICAL_DIR, "data/canonical", required=True, description="Canonical clean and attacked JSONL files.")
if UPLOAD_DATA_VIEWS:
    add_source_folder(VIEWS_DIR, "data/views", required=True, description="Training and evaluation views used by the scenarios.")
if UPLOAD_CONFIGS:
    add_source_folder(CONFIG_DIR, "configs", required=False, description="Experiment and training configuration files.")
if UPLOAD_DATA_MANIFESTS:
    add_source_folder(MANIFESTS_DIR / "data", "manifests/data", required=False, description="Dataset creation manifests.")
    add_source_folder(MANIFESTS_DIR / "environment", "manifests/environment", required=False, description="Environment setup manifests.")

print("Pastas fonte:")
for item in source_folders:
    print("-", item["source_path"], "->", item["target_path"])

print("\nFontes ausentes opcionais:")
for item in missing_sources:
    print("-", item["source_path"], "->", item["target_path"], "required=", item["required"])


Pastas fonte:
- /workspace/pi-defense-exp/data/canonical -> data/canonical
- /workspace/pi-defense-exp/data/views -> data/views
- /workspace/pi-defense-exp/configs -> configs
- /workspace/pi-defense-exp/manifests/data -> manifests/data
- /workspace/pi-defense-exp/manifests/environment -> manifests/environment

Fontes ausentes opcionais:


## 9. Índice local dos arquivos que entrarão no upload

Antes de enviar qualquer coisa, o notebook constrói um índice dos arquivos que serão enviados. Esse índice serve para auditoria e também ajuda a verificar se nenhum arquivo indesejado entrou no escopo do upload.

Arquivos de cache, checkpoints do Jupyter e caches Python são ignorados.


In [8]:
IGNORE_DIR_NAMES = {".ipynb_checkpoints", "__pycache__", ".cache"}
IGNORE_FILE_SUFFIXES = {".pyc", ".pyo"}


def should_include_file(path: Path) -> bool:
    if set(path.parts).intersection(IGNORE_DIR_NAMES):
        return False
    if path.suffix in IGNORE_FILE_SUFFIXES:
        return False
    return True


file_index_rows = []

for folder_item in source_folders:
    source_root = folder_item["source_path"]
    target_root = folder_item["target_path"]
    for path in sorted(source_root.rglob("*")):
        if not path.is_file() or not should_include_file(path):
            continue
        relative_to_source = path.relative_to(source_root)
        target_path = str(Path(target_root) / relative_to_source).replace("\\", "/")
        line_count = count_jsonl_lines(path) if path.suffix == ".jsonl" else None
        file_index_rows.append({
            "source_path": str(path),
            "target_path": target_path,
            "source_group": target_root,
            "size_mb": file_size_mb(path),
            "line_count": line_count,
        })

file_index_df = pd.DataFrame(file_index_rows)
if file_index_df.empty:
    raise RuntimeError("Nenhum arquivo encontrado para upload.")
file_index_df = file_index_df.sort_values(["target_path"]).reset_index(drop=True)

display(file_index_df.head(20))
print("Total de arquivos candidatos:", len(file_index_df))
print("Tamanho total estimado MB:", round(file_index_df["size_mb"].sum(), 2))


,source_path,target_path,source_group,size_mb,line_count
0,/workspace/pi-defense-exp/configs/experiment.yaml,configs/experiment.yaml,configs,0.000406,NaN
1,/workspace/pi-defense-exp/configs/training_pla...,configs/training_plan.yaml,configs,0.008863,NaN
2,/workspace/pi-defense-exp/data/canonical/test_...,data/canonical/test_attacked_seen.jsonl,data/canonical,7.323574,9380.0
3,/workspace/pi-defense-exp/data/canonical/test_...,data/canonical/test_attacked_unseen.jsonl,data/canonical,4.618910,5628.0
4,/workspace/pi-defense-exp/data/canonical/test_...,data/canonical/test_clean.jsonl,data/canonical,0.894280,1876.0
5,/workspace/pi-defense-exp/data/canonical/train...,data/canonical/train_attacked_seen.jsonl,data/canonical,1.636912,2100.0
6,/workspace/pi-defense-exp/data/canonical/train...,data/canonical/train_clean.jsonl,data/canonical,1.000756,2100.0
7,/workspace/pi-defense-exp/data/canonical/valid...,data/canonical/validation_attacked_seen.jsonl,data/canonical,0.278545,350.0
8,/workspace/pi-defense-exp/data/canonical/valid...,data/canonical/validation_clean.jsonl,data/canonical,0.170476,350.0
9,/workspace/pi-defense-exp/data/views/evaluatio...,data/views/evaluation/test_attacked_seen.jsonl,data/views,7.323574,9380.0


Total de arquivos candidatos: 22
Tamanho total estimado MB: 36.66


## 10. Validação das contagens principais

Esta etapa valida os arquivos esperados do dataset canônico e das views principais. A validação não substitui o manifesto do notebook 02, mas ajuda a evitar upload incompleto.


In [9]:
expected_line_counts = {
    "data/canonical/train_clean.jsonl": 2100,
    "data/canonical/validation_clean.jsonl": 350,
    "data/canonical/test_clean.jsonl": 1876,
    "data/canonical/train_attacked_seen.jsonl": 2100,
    "data/canonical/validation_attacked_seen.jsonl": 350,
    "data/canonical/test_attacked_seen.jsonl": 9380,
    "data/canonical/test_attacked_unseen.jsonl": 5628,
    "data/views/struq/train_sft.jsonl": 4200,
    "data/views/struq/validation_sft.jsonl": 700,
    "data/views/secalign/train_dpo.jsonl": 2100,
    "data/views/secalign/validation_dpo.jsonl": 350,
    "data/views/ih/train_sft.jsonl": 4200,
    "data/views/ih/validation_sft.jsonl": 700,
    "data/views/evaluation/test_clean.jsonl": 1876,
    "data/views/evaluation/test_attacked_seen.jsonl": 9380,
    "data/views/evaluation/test_attacked_unseen.jsonl": 5628,
}

validation_rows = []
for target_path, expected_count in expected_line_counts.items():
    matches = file_index_df[file_index_df["target_path"] == target_path]
    if matches.empty:
        validation_rows.append({"target_path": target_path, "expected": expected_count, "observed": None, "status": "missing"})
        continue
    observed_count = int(matches.iloc[0]["line_count"])
    status = "ok" if observed_count == expected_count else "count_mismatch"
    validation_rows.append({"target_path": target_path, "expected": expected_count, "observed": observed_count, "status": status})

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

bad_rows = validation_df[validation_df["status"] != "ok"]
if not bad_rows.empty:
    raise RuntimeError("Alguns arquivos obrigatórios estão ausentes ou com contagem inesperada.")

print("Validação das contagens principais concluída.")


,target_path,expected,observed,status
0,data/canonical/train_clean.jsonl,2100,2100,ok
1,data/canonical/validation_clean.jsonl,350,350,ok
2,data/canonical/test_clean.jsonl,1876,1876,ok
3,data/canonical/train_attacked_seen.jsonl,2100,2100,ok
4,data/canonical/validation_attacked_seen.jsonl,350,350,ok
5,data/canonical/test_attacked_seen.jsonl,9380,9380,ok
6,data/canonical/test_attacked_unseen.jsonl,5628,5628,ok
7,data/views/struq/train_sft.jsonl,4200,4200,ok
8,data/views/struq/validation_sft.jsonl,700,700,ok
9,data/views/secalign/train_dpo.jsonl,2100,2100,ok


Validação das contagens principais concluída.


## 11. Geração do dataset card

O dataset card é o `README.md` do repositório no Hugging Face. Ele explica o objetivo do dataset, tasks, estrutura dos arquivos, ataques, limitações e aviso de conteúdo sensível.

Como este dataset é derivado de múltiplas fontes e inclui transformações experimentais, a licença é indicada como `other` no bloco YAML. Antes de tornar o dataset público, recomenda-se revisar manualmente as licenças dos datasets originais e ajustar o card conforme necessário.


In [10]:
def make_dataset_card() -> str:
    return """---
license: other
language:
- en
task_categories:
- text-classification
tags:
- prompt-injection
- llm-safety
- adversarial-evaluation
- instruction-following
- text-classification
size_categories:
- 10K<n<100K
pretty_name: Prompt Injection Defense Experiment Dataset
---

# Prompt Injection Defense Experiment Dataset

This dataset contains the processed data used in an experimental evaluation of prompt-injection defenses for instruction-following language models.

The dataset is organized around classification tasks, clean examples, attacked examples, and scenario-specific training/evaluation views.

## Intended Use

This dataset is intended for academic and experimental evaluation of prompt-injection defenses. It can be used to reproduce the data stage of the experiment and to train/evaluate defense scenarios such as:

- StruQ-like supervised fine-tuning
- SecAlign-like DPO preference optimization
- Instruction-Hierarchy-like supervised fine-tuning

## Content Warning

This dataset may contain offensive or sensitive text because it includes examples derived from hate/offensive language classification data. It also contains synthetic prompt-injection templates inserted into untrusted data fields.

The dataset should be used only for research, evaluation, and defensive analysis.

## Tasks

| Task | Description |
|---|---|
| `mrpc` | Paraphrase classification |
| `rte` | Textual entailment classification |
| `cola` | Grammatical acceptability classification |
| `qqp` | Duplicate question classification |
| `sst2` | Sentiment classification |
| `sms_spam` | SMS spam classification |
| `hsol` | Hate/offensive/neutral classification |

## Dataset Structure

```text
data/canonical/
data/views/
```

### Canonical files

```text
data/canonical/train_clean.jsonl
data/canonical/validation_clean.jsonl
data/canonical/test_clean.jsonl
data/canonical/train_attacked_seen.jsonl
data/canonical/validation_attacked_seen.jsonl
data/canonical/test_attacked_seen.jsonl
data/canonical/test_attacked_unseen.jsonl
```

### Training and evaluation views

```text
data/views/struq/train_sft.jsonl
data/views/struq/validation_sft.jsonl

data/views/secalign/train_dpo.jsonl
data/views/secalign/validation_dpo.jsonl

data/views/ih/train_sft.jsonl
data/views/ih/validation_sft.jsonl

data/views/evaluation/test_clean.jsonl
data/views/evaluation/test_attacked_seen.jsonl
data/views/evaluation/test_attacked_unseen.jsonl
```

## Attacks

Seen attacks used in training/validation:

```text
naive
ignore
escape
fake_comp
combine
```

Unseen/adaptive attacks used in testing:

```text
combine_adaptive
gcg
gcg_adaptive
```

In this version, `gcg` and `gcg_adaptive` are represented as GCG-like templates, not optimized adversarial suffixes.

## Splits and Counts

| File | Rows |
|---|---:|
| `train_clean.jsonl` | 2,100 |
| `validation_clean.jsonl` | 350 |
| `test_clean.jsonl` | 1,876 |
| `train_attacked_seen.jsonl` | 2,100 |
| `validation_attacked_seen.jsonl` | 350 |
| `test_attacked_seen.jsonl` | 9,380 |
| `test_attacked_unseen.jsonl` | 5,628 |

## Canonical Schema

A canonical clean row contains fields such as:

```json
{
  "id": "...",
  "task_name": "...",
  "split": "...",
  "trusted_instruction": "...",
  "clean_input": "...",
  "expected_answer": "...",
  "label_space": ["..."]
}
```

An attacked row additionally contains fields such as:

```json
{
  "untrusted_data": "...",
  "attack_type": "...",
  "attack_target": "...",
  "seen_in_training": true
}
```

## Limitations

- The GCG attacks are GCG-like templates, not fully optimized adversarial suffixes.
- The dataset focuses on classification tasks and may not generalize to open-ended generation tasks.
- The dataset is designed for defensive prompt-injection evaluation, not for deploying attacks.
- Some source datasets may have their own license and usage constraints; users should review original dataset licenses before redistribution or public release.

## Generated by

This dataset card was generated by `12_upload_dataset_to_huggingface.ipynb`.

Upload mode:

```text
No intermediate heavy copy. Files are uploaded directly from their original project paths.
```
"""


dataset_card_path = HF_DATASET_EXPORT_DIR / "README.md"
dataset_card_text = make_dataset_card()
dataset_card_path.write_text(dataset_card_text, encoding="utf-8")

print("Dataset card criado em:", dataset_card_path)


Dataset card criado em: /workspace/pi-defense-exp/exports/huggingface_dataset_upload/README.md


## 12. Manifesto local do upload

Antes do upload, o notebook cria um manifesto local com repositório alvo, modo dry-run, fontes incluídas, arquivos candidatos, contagens principais, fontes opcionais ausentes e confirmação de que não há cópia intermediária pesada.


In [11]:
file_index_csv_path = HF_DATASET_EXPORT_DIR / "file_index.csv"
file_index_json_path = HF_DATASET_EXPORT_DIR / "file_index.json"
missing_sources_path = HF_DATASET_EXPORT_DIR / "missing_optional_sources.json"
upload_manifest_path = HF_DATASET_EXPORT_DIR / "experiment_dataset_upload_manifest.json"

file_index_df.to_csv(file_index_csv_path, index=False)
write_json(file_index_json_path, file_index_df.to_dict(orient="records"))
write_json(missing_sources_path, {"missing_sources": missing_sources})

upload_manifest = {
    "notebook": "12_upload_dataset_to_huggingface",
    "created_at_utc": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "dataset_repo_id": DATASET_REPO_ID,
    "repo_type": "dataset",
    "dry_run": DRY_RUN,
    "private_repo": PRIVATE_REPO,
    "no_intermediate_heavy_copy": True,
    "local_metadata_dir": str(HF_DATASET_EXPORT_DIR),
    "included_source_folders": [{"source_path": str(item["source_path"]), "target_path": item["target_path"], "description": item["description"]} for item in source_folders],
    "included_metadata_files": [str(dataset_card_path), str(file_index_csv_path), str(file_index_json_path), str(missing_sources_path)],
    "file_count": int(len(file_index_df)),
    "estimated_total_size_mb": float(file_index_df["size_mb"].sum()),
    "expected_line_counts": expected_line_counts,
    "validation": validation_df.to_dict(orient="records"),
    "missing_sources": missing_sources,
    "excluded_by_policy": ["data/cache/", ".venv/", "adapters/", "logs/", "results/", "model weights and Hugging Face caches"],
}

write_json(upload_manifest_path, upload_manifest)

print("Manifesto local criado em:", upload_manifest_path)
print("Índice CSV criado em:", file_index_csv_path)
print("Índice JSON criado em:", file_index_json_path)


Manifesto local criado em: /workspace/pi-defense-exp/exports/huggingface_dataset_upload/experiment_dataset_upload_manifest.json
Índice CSV criado em: /workspace/pi-defense-exp/exports/huggingface_dataset_upload/file_index.csv
Índice JSON criado em: /workspace/pi-defense-exp/exports/huggingface_dataset_upload/file_index.json


## 13. Resumo do dry-run

Esta etapa mostra o que será enviado. Se `DRY_RUN=True`, o notebook valida e prepara metadados locais, mas não executa nenhuma operação remota.


In [12]:
summary = {
    "dataset_repo_id": DATASET_REPO_ID,
    "dry_run": DRY_RUN,
    "private_repo": PRIVATE_REPO,
    "source_folders": len(source_folders),
    "candidate_files": int(len(file_index_df)),
    "estimated_total_size_mb": float(file_index_df["size_mb"].sum()),
    "metadata_dir": str(HF_DATASET_EXPORT_DIR),
}

write_json(HF_DATASET_EXPORT_DIR / "dry_run_summary.json", summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))

if DRY_RUN:
    print("\nDRY_RUN=True: nenhuma operação remota será executada nesta execução.")


{
  "dataset_repo_id": "leinha/pi-defense-experiment-dataset",
  "dry_run": false,
  "private_repo": true,
  "source_folders": 5,
  "candidate_files": 22,
  "estimated_total_size_mb": 36.66295623779297,
  "metadata_dir": "/workspace/pi-defense-exp/exports/huggingface_dataset_upload"
}


## 14. Criar repositório remoto

Quando `DRY_RUN=False`, esta etapa cria ou reutiliza o repositório remoto no Hugging Face Hub.

O repositório é criado com:

```python
repo_type="dataset"
```

Por padrão, o repositório é privado.


In [13]:
if not DRY_RUN:
    log_event("dataset_repo_create_started", {"repo_id": DATASET_REPO_ID, "private": PRIVATE_REPO})
    create_repo(repo_id=DATASET_REPO_ID, repo_type="dataset", private=PRIVATE_REPO, exist_ok=True)
    log_event("dataset_repo_create_completed", {"repo_id": DATASET_REPO_ID, "private": PRIVATE_REPO})
    print("Repositório dataset criado/reutilizado:", DATASET_REPO_ID)
else:
    print("DRY_RUN=True: criação de repositório remoto pulada.")


Repositório dataset criado/reutilizado: leinha/pi-defense-experiment-dataset


## 15. Upload dos metadados leves

Esta etapa envia os arquivos leves gerados localmente: `README.md`, manifesto, índice de arquivos, fontes opcionais ausentes e resumo de dry-run.


In [14]:
metadata_files_to_upload = [
    {"local_path": dataset_card_path, "path_in_repo": "README.md"},
    {"local_path": upload_manifest_path, "path_in_repo": "metadata/experiment_dataset_upload_manifest.json"},
    {"local_path": file_index_csv_path, "path_in_repo": "metadata/file_index.csv"},
    {"local_path": file_index_json_path, "path_in_repo": "metadata/file_index.json"},
    {"local_path": missing_sources_path, "path_in_repo": "metadata/missing_optional_sources.json"},
    {"local_path": HF_DATASET_EXPORT_DIR / "dry_run_summary.json", "path_in_repo": "metadata/dry_run_summary.json"},
]

if not DRY_RUN:
    for item in metadata_files_to_upload:
        log_event("metadata_upload_started", {"repo_id": DATASET_REPO_ID, "local_path": str(item["local_path"]), "path_in_repo": item["path_in_repo"]})
        upload_file(
            repo_id=DATASET_REPO_ID,
            repo_type="dataset",
            path_or_fileobj=str(item["local_path"]),
            path_in_repo=item["path_in_repo"],
            commit_message=f"Upload metadata: {item['path_in_repo']}",
        )
        log_event("metadata_upload_completed", {"repo_id": DATASET_REPO_ID, "local_path": str(item["local_path"]), "path_in_repo": item["path_in_repo"]})
    print("Metadados leves enviados.")
else:
    print("DRY_RUN=True: upload de metadados pulado.")
    for item in metadata_files_to_upload:
        print("Seria enviado:", item["local_path"], "->", item["path_in_repo"])


Metadados leves enviados.


## 16. Upload direto das pastas de dataset

Esta é a etapa principal do notebook.

Cada pasta é enviada diretamente de sua origem no projeto para o repositório remoto. Não existe cópia intermediária para `exports/`.

Exemplo conceitual:

```text
/workspace/pi-defense-exp/data/canonical  ->  data/canonical no Hub
/workspace/pi-defense-exp/data/views      ->  data/views no Hub
```

Esse desenho evita duplicar dados localmente e reduz o uso de disco.


In [15]:
folder_upload_records = []

ignore_patterns = [
    "**/.ipynb_checkpoints/**",
    "**/__pycache__/**",
    "**/*.pyc",
    "**/*.pyo",
]

if not DRY_RUN:
    for item in source_folders:
        started_at = utc_now()
        start_time = time.time()
        log_event("source_folder_upload_started", {"repo_id": DATASET_REPO_ID, "source_path": str(item["source_path"]), "path_in_repo": item["target_path"]})
        try:
            upload_folder(
                repo_id=DATASET_REPO_ID,
                repo_type="dataset",
                folder_path=str(item["source_path"]),
                path_in_repo=item["target_path"],
                ignore_patterns=ignore_patterns,
                commit_message=f"Upload dataset folder: {item['target_path']}",
            )
            elapsed_seconds = time.time() - start_time
            record = {
                "source_path": str(item["source_path"]),
                "path_in_repo": item["target_path"],
                "status": "completed",
                "started_at_utc": started_at,
                "finished_at_utc": utc_now(),
                "elapsed_seconds": elapsed_seconds,
            }
            folder_upload_records.append(record)
            log_event("source_folder_upload_completed", record)
            print("Upload concluído:", item["source_path"], "->", item["target_path"])
        except Exception as error:
            error_text = traceback.format_exc()
            error_path = HF_DATASET_LOG_DIR / f"upload_error_{item['target_path'].replace('/', '_')}.txt"
            error_path.write_text(error_text, encoding="utf-8")
            log_event("source_folder_upload_failed", {"repo_id": DATASET_REPO_ID, "source_path": str(item["source_path"]), "path_in_repo": item["target_path"], "error": repr(error), "error_path": str(error_path)})
            print("Falha no upload da pasta:", item["source_path"])
            print("Erro registrado em:", error_path)
            raise
else:
    print("DRY_RUN=True: upload das pastas de dataset pulado.")
    for item in source_folders:
        print("Seria enviado:", item["source_path"], "->", item["target_path"])

folder_upload_df = pd.DataFrame(folder_upload_records)
display(folder_upload_df)


Upload concluído: /workspace/pi-defense-exp/data/canonical -> data/canonical
Upload concluído: /workspace/pi-defense-exp/data/views -> data/views
Upload concluído: /workspace/pi-defense-exp/configs -> configs
Upload concluído: /workspace/pi-defense-exp/manifests/data -> manifests/data
Upload concluído: /workspace/pi-defense-exp/manifests/environment -> manifests/environment


,source_path,path_in_repo,status,started_at_utc,finished_at_utc,elapsed_seconds
0,/workspace/pi-defense-exp/data/canonical,data/canonical,completed,2026-07-02T03:55:02.483043+00:00,2026-07-02T03:55:04.465755+00:00,1.982695
1,/workspace/pi-defense-exp/data/views,data/views,completed,2026-07-02T03:55:04.466069+00:00,2026-07-02T03:55:06.377503+00:00,1.911422
2,/workspace/pi-defense-exp/configs,configs,completed,2026-07-02T03:55:06.377809+00:00,2026-07-02T03:55:06.725944+00:00,0.348126
3,/workspace/pi-defense-exp/manifests/data,manifests/data,completed,2026-07-02T03:55:06.726152+00:00,2026-07-02T03:55:07.090567+00:00,0.364405
4,/workspace/pi-defense-exp/manifests/environment,manifests/environment,completed,2026-07-02T03:55:07.090782+00:00,2026-07-02T03:55:07.447603+00:00,0.356813


## 17. Verificação remota

Após o upload real, esta etapa consulta o repositório remoto e lista os arquivos encontrados. Isso ajuda a conferir se os arquivos principais foram enviados para os caminhos esperados.


In [16]:
remote_files = []

if not DRY_RUN:
    api = HfApi()
    remote_files = api.list_repo_files(repo_id=DATASET_REPO_ID, repo_type="dataset")
    remote_files_df = pd.DataFrame({"path_in_repo": remote_files})
    display(remote_files_df.head(50))

    required_remote_paths = [
        "README.md",
        "metadata/experiment_dataset_upload_manifest.json",
        "data/canonical/train_clean.jsonl",
        "data/canonical/test_attacked_seen.jsonl",
        "data/views/struq/train_sft.jsonl",
        "data/views/secalign/train_dpo.jsonl",
        "data/views/ih/train_sft.jsonl",
        "data/views/evaluation/test_clean.jsonl",
    ]
    missing_remote = [path for path in required_remote_paths if path not in remote_files]
    if missing_remote:
        raise RuntimeError(f"Arquivos remotos obrigatórios ausentes: {missing_remote}")
    print("Verificação remota concluída.")
else:
    print("DRY_RUN=True: verificação remota pulada.")


,path_in_repo
0,.gitattributes
1,LICENSE
2,README.md
3,configs/experiment.yaml
4,configs/training_plan.yaml
5,data/canonical/test_attacked_seen.jsonl
6,data/canonical/test_attacked_unseen.jsonl
7,data/canonical/test_clean.jsonl
8,data/canonical/train_attacked_seen.jsonl
9,data/canonical/train_clean.jsonl


Verificação remota concluída.


## 18. Manifesto final do notebook 12

Esta etapa gera os manifestos finais do upload de dataset.

Os manifestos registram repositório alvo, modo de execução, fontes enviadas, arquivos candidatos, fontes ausentes opcionais, caminhos de logs e metadados locais e arquivos remotos verificados, quando aplicável.


In [17]:
final_manifest_json_path = HF_DATASET_MANIFEST_DIR / "12_upload_dataset_to_huggingface_manifest.json"
final_manifest_md_path = HF_DATASET_MANIFEST_DIR / "12_upload_dataset_to_huggingface_manifest.md"

final_manifest = {
    "notebook": "12_upload_dataset_to_huggingface",
    "created_at_utc": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "dataset_repo_id": DATASET_REPO_ID,
    "repo_type": "dataset",
    "dry_run": DRY_RUN,
    "private_repo": PRIVATE_REPO,
    "acknowledge_dataset_content": ACKNOWLEDGE_DATASET_CONTENT,
    "no_intermediate_heavy_copy": True,
    "local_metadata_dir": str(HF_DATASET_EXPORT_DIR),
    "log_dir": str(HF_DATASET_LOG_DIR),
    "manifest_dir": str(HF_DATASET_MANIFEST_DIR),
    "source_folders": [{"source_path": str(item["source_path"]), "target_path": item["target_path"], "description": item["description"]} for item in source_folders],
    "file_count": int(len(file_index_df)),
    "estimated_total_size_mb": float(file_index_df["size_mb"].sum()),
    "folder_upload_records": folder_upload_records,
    "remote_file_count": len(remote_files),
    "remote_files_sample": remote_files[:50],
    "missing_sources": missing_sources,
    "metadata_files": [str(dataset_card_path), str(upload_manifest_path), str(file_index_csv_path), str(file_index_json_path), str(missing_sources_path)],
}

write_json(final_manifest_json_path, final_manifest)

source_table_lines = ["| Source | Target in repo | Required | Description |", "|---|---|---:|---|"]
for item in source_folders:
    source_table_lines.append(f"| `{item['source_path']}` | `{item['target_path']}` | {item['required']} | {item['description']} |")

missing_table_lines = ["| Source | Target in repo | Required | Reason |", "|---|---|---:|---|"]
if missing_sources:
    for item in missing_sources:
        missing_table_lines.append(f"| `{item['source_path']}` | `{item['target_path']}` | {item['required']} | {item['reason']} |")
else:
    missing_table_lines.append("| _None_ | _None_ |  | _None_ |")

final_manifest_md = f"""# Manifesto — Upload do dataset para Hugging Face

## Identificação

- Notebook: `12_upload_dataset_to_huggingface`
- Gerado em UTC: `{final_manifest['created_at_utc']}`
- Dataset repo: `{DATASET_REPO_ID}`
- Repo type: `dataset`
- Dry run: `{DRY_RUN}`
- Private repo: `{PRIVATE_REPO}`
- Sem cópia intermediária pesada: `True`

## Fontes incluídas

{chr(10).join(source_table_lines)}

## Fontes opcionais ausentes

{chr(10).join(missing_table_lines)}

## Resumo

- Arquivos candidatos: `{len(file_index_df)}`
- Tamanho total estimado: `{file_index_df['size_mb'].sum():.2f} MB`
- Diretório local de metadados leves: `{HF_DATASET_EXPORT_DIR}`
- Diretório de logs: `{HF_DATASET_LOG_DIR}`

## Observações

- Este notebook não copia os dados para `exports/` antes do upload.
- Os arquivos são enviados diretamente dos diretórios originais do projeto.
- `data/cache/`, `.venv/`, adaptadores, logs e resultados não fazem parte deste upload de dataset.
- O dataset pode conter conteúdo sensível de HSOL e templates de prompt injection.
- Recomenda-se revisar o dataset card antes de tornar o repositório público.
"""

final_manifest_md_path.write_text(final_manifest_md, encoding="utf-8")

print("Manifesto JSON:", final_manifest_json_path)
print("Manifesto Markdown:", final_manifest_md_path)


Manifesto JSON: /workspace/pi-defense-exp/manifests/huggingface_dataset_upload/12_upload_dataset_to_huggingface_manifest.json
Manifesto Markdown: /workspace/pi-defense-exp/manifests/huggingface_dataset_upload/12_upload_dataset_to_huggingface_manifest.md


## 19. Próximos passos

Depois de rodar este notebook em modo real (`DRY_RUN=False`), revise o repositório de dataset no Hugging Face.

Verifique principalmente:

```text
- se o README.md aparece como dataset card;
- se os arquivos JSONL estão nos caminhos esperados;
- se os metadados foram enviados;
- se o repositório está privado, caso ainda precise de revisão;
- se não há tokens, caches, adaptadores ou arquivos grandes indevidos;
- se o aviso de conteúdo sensível está claro.
```

Caso o dataset seja publicado, recomenda-se também revisar as licenças e termos de uso das fontes originais utilizadas na composição da base experimental.
